In [7]:
from supabase import create_client
import streamlit as st

supabase = create_client(
    st.secrets["SUPABASE_URL"],
    st.secrets["SUPABASE_KEY"]
)


In [8]:
res = supabase.table("reports").select("*").execute()
st.write(res.data)


2026-08-25 01:15:03.414 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-25 01:15:03.611 
  command:

    streamlit run c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-08-25 01:15:03.611 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-25 01:15:03.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [ ]:
import streamlit as st
from supabase import create_client, Client
import datetime
import uuid

# Supabase 接続
url = st.secrets["https://vwibdnfyjhlkhtguzagf.supabase.co"]
key = st.secrets["sb_publishable_ZXjYOKOuzTgTbtkIKaQ3cA_zC7O1KK_"]
supabase: Client = create_client(url, key)

st.title("📷 災害被害を報告する")

# 位置情報入力
lat = st.number_input("緯度", format="%.6f")
lon = st.number_input("経度", format="%.6f")

# 被害種別
damage_type = st.selectbox(
    "被害の種類",
    ["道路", "建物", "ライフライン", "その他"]
)

# コメント
comment = st.text_area("状況説明")

# 写真アップロード
photo = st.file_uploader("写真をアップロード", type=["jpg", "jpeg", "png"])

if st.button("報告する"):
    if photo is not None:
        # 写真を Supabase Storage に保存
        file_id = str(uuid.uuid4())
        file_path = f"reports/{file_id}.jpg"
        supabase.storage.from_("photos").upload(file_path, photo)

        # DB に記録
        data = {
            "lat": lat,
            "lon": lon,
            "damage_type": damage_type,
            "comment": comment,
            "photo_url": file_path,
            "created_at": datetime.datetime.utcnow().isoformat()
        }
        supabase.table("reports").insert(data).execute()

        st.success("報告を送信しました！")
    else:
        st.error("写真をアップロードしてください")


In [ ]:
import streamlit as st
from supabase import create_client, Client
import pandas as pd
import pydeck as pdk

# Supabase 接続
url = st.secrets["SUPABASE_URL"]
key = st.secrets["SUPABASE_KEY"]
supabase: Client = create_client(url, key)

st.title("🗺️ 被害マップ")

# DB からデータ取得
res = supabase.table("reports").select("*").execute()
df = pd.DataFrame(res.data)

# 地図プロット
layer = pdk.Layer(
    "ScatterplotLayer",
    df,
    get_position=["lon", "lat"],
    get_color=[255, 0, 0],
    get_radius=50,
)

view_state = pdk.ViewState(
    latitude=df["lat"].mean() if len(df) else 35.0,
    longitude=df["lon"].mean() if len(df) else 135.0,
    zoom=10,
)

st.pydeck_chart(pdk.Deck(layers=[layer], initial_view_state=view_state))

# データ一覧
st.dataframe(df)
